# Evaluating and monitoring forecast performance: lab notebook

**Statistical Modelling and Forecasting, Statistics background for forecasting**

Run the cells from top to bottom (Runtime > Run all). NumPy, pandas, SciPy, statsmodels and scikit-learn are already installed in Colab. Read each section, run it, and compare your output with the expected output at the end of the notebook.

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # keeps the output tidy

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.holtwinters import ExponentialSmoothing, SimpleExpSmoothing

### 1. The numerical problem: error measures by hand

In [ ]:
# 1. The numerical problem: error measures by hand
actual = np.array([100, 110, 120, 95, 130], dtype=float)
fc = np.array([105, 100, 125, 90, 128], dtype=float)
e = actual - fc
mae, rmse = np.mean(np.abs(e)), np.sqrt(np.mean(e ** 2))
mape = 100 * np.mean(np.abs(e) / actual)
print("ME", e.mean(), "| MAE", mae, "| RMSE", round(rmse, 3), "| MAPE", round(mape, 2))
print("MASE with naive scale 5:", mae / 5, "| tracking signal:", round(e.sum() / mae, 3))

### 2. Real data: monthly CO2, chronological split

In [ ]:
# 2. Real data: monthly CO2, chronological split
co2 = sm.datasets.co2.load_pandas().data["co2"].resample("MS").mean().interpolate()
train, test = co2.iloc[:-48], co2.iloc[-48:]
print("train:", train.index[0].date(), "to", train.index[-1].date(), "| test:", test.index[0].date(), "to", test.index[-1].date())

### 3. Rolling-origin one-step forecasts for four simple methods

In [ ]:
# 3. Rolling-origin one-step forecasts for four simple methods
def one_step(method, y, t):
    if method == "naive": return y[t]
    if method == "seasonal naive": return y[t + 1 - 12]
    if method == "drift": return y[t] + (y[t] - y[0]) / t
    if method == "mean": return y[:t + 1].mean()

y = co2.values
n_train = len(train)
scale = np.mean(np.abs(np.diff(y[:n_train])))                  # in-sample naive MAE
for m in ["naive", "seasonal naive", "drift", "mean"]:
    errs = np.array([y[t + 1] - one_step(m, y, t) for t in range(n_train - 1, len(y) - 1)])
    print(f"{m:15s} MAE={np.mean(np.abs(errs)):6.3f} RMSE={np.sqrt(np.mean(errs**2)):6.3f} MASE={np.mean(np.abs(errs)) / scale:6.2f} ME={errs.mean():6.3f}")

### 4. A model with a fit: Holt-Winters on the training data, forecast 48 months

In [ ]:
# 4. A model with a fit: Holt-Winters on the training data, forecast 48 months
hw = ExponentialSmoothing(train, trend="add", seasonal="add", seasonal_periods=12).fit()
f = hw.forecast(48)
e48 = test.values - f.values
print("Holt-Winters 48-step: MAE", round(np.mean(np.abs(e48)), 3), "| RMSE", round(np.sqrt(np.mean(e48 ** 2)), 3), "| ME", round(e48.mean(), 3))

### 5. Monitoring: CUSUM of standardised errors with and without a level shift of +2 sigma

In [ ]:
# 5. Monitoring: CUSUM of standardised errors with and without a level shift of +2 sigma
rng = np.random.default_rng(1)
z0 = rng.normal(0, 1, 60)
# (with h = 5 a false alarm happens about once in 465 periods on average)
for label, shift in [("no shift", 0.0), ("level shift +2 sigma", 2.0)]:
    z = z0 + shift
    cp = cm = 0.0
    alarm = None
    for i, q in enumerate(z, start=1):
        cp = max(0.0, cp + q - 0.5)
        cm = max(0.0, cm - q - 0.5)
        if alarm is None and (cp > 5 or cm > 5):
            alarm = i
    print(f"{label:22s} first CUSUM alarm at period:", alarm)
tracking = np.cumsum(z0) / (np.cumsum(np.abs(z0)) / np.arange(1, 61))
print("plain tracking signal of pure noise at periods 6 and 60:", round(tracking[5], 2), round(tracking[59], 2), "(grows like sqrt(n), so use it over short spans)")

## Expected output

Running all cells above should print:

```
ME 1.4 | MAE 5.4 | RMSE 5.983 | MAPE 5.01
MASE with naive scale 5: 1.08 | tracking signal: 1.296
train: 1958-03-01 to 1997-12-01 | test: 1998-01-01 to 2001-12-01
naive           MAE= 1.080 RMSE= 1.215 MASE=  1.03 ME= 0.141
seasonal naive  MAE= 1.788 RMSE= 1.977 MASE=  1.70 ME= 1.788
drift           MAE= 1.051 RMSE= 1.209 MASE=  1.00 ME= 0.036
mean            MAE=30.627 RMSE=30.696 MASE= 29.17 ME=30.627
Holt-Winters 48-step: MAE 1.458 | RMSE 1.507 | ME 1.458
no shift               first CUSUM alarm at period: None
level shift +2 sigma   first CUSUM alarm at period: 3
plain tracking signal of pure noise at periods 6 and 60: 2.23 -4.46 (grows like sqrt(n), so use it over short spans)
```

## Practice exercises

Try these in the cells below.

1. Compare naive, seasonal naive and drift on the seasonal series. Which gives the lowest MASE?
2. Set the SES smoothing weight α to 0.1 and to 0.9 and see which is better on this series.
3. Tick "Level shift at start of test" and watch the CUSUM cross the alarm line. How many periods pass before it flags?

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3
